# 01 Data preparation

## Load data

In [1]:
import numpy as np
import pandas as pd
import pyreadstat

RAW_SAV = "../../data/BDIR81SV/BDIR81FL.SAV"

In [2]:
RENAME = {
    # survey design
    "V005": "weight", "V021": "psu", "V022": "strata", "SQTYPE": "questionnaire",
    # fertility preference
    "V605": "fert_pref", "V613": "ideal_children",
    # sociodemographic
    "V012": "age", "V024": "division", "V025": "residence", "V130": "religion",
    "V136": "hh_size", "V151": "hh_head_sex", "V106": "edu_level", "V190": "wealth_quintile",
    "V714": "currently_working",
    # residence history
    "V104": "years_in_residence", "V105": "previous_residence", "V175": "reason_moved",
    # media
    "V157": "reads_newspaper", "V158": "listens_radio", "V159": "watches_tv",
    # digital inclusion
    "V169A": "owns_mobile", "V169C": "is_smartphone", "V170": "bank_account",
    "V171A": "used_internet_12m", "V177": "bank_transaction_12m",
    # marriage
    "V501": "marital_status", "V504": "husband_lives_with", "V511": "age_first_marriage",
    "V512": "years_since_cohabitation",
    # reproductive history and contraception
    "V206": "sons_died", "V207": "daughters_died", "V213": "currently_pregnant",
    "V218": "living_children", "V312": "contra_method", "V362": "intent_use_contra",
    # husband
    "V701": "husband_edu", "V704": "husband_occupation",
    # decision-making
    "V743A": "dec_own_health", "V743B": "dec_large_purchase", "V743D": "dec_visit_family",
    # attitudes toward wife beating
    "V744A": "beat_goes_out", "V744B": "beat_neglects_kids", "V744C": "beat_argues",
    "V744D": "beat_refuses_sex", "V744E": "beat_burns_food",
    # GAD-7 (anxiety) and PHQ-9 (depression)
    "MTH1": "gad_1", "MTH2": "gad_2", "MTH3": "gad_3", "MTH4": "gad_4",
    "MTH5": "gad_5", "MTH6": "gad_6", "MTH7": "gad_7",
    "MTH8": "phq_1", "MTH9": "phq_2", "MTH10": "phq_3", "MTH11": "phq_4", "MTH12": "phq_5",
    "MTH13": "phq_6", "MTH14": "phq_7", "MTH15": "phq_8", "MTH16": "phq_9",
}
GAD_COLS = [f"gad_{i}" for i in range(1, 8)]
PHQ_COLS = [f"phq_{i}" for i in range(1, 10)]

raw, meta = pyreadstat.read_sav(RAW_SAV, usecols=list(RENAME))
raw = raw.rename(columns=RENAME)
raw.shape

(30078, 62)

## Sample selection and outcome

In [3]:
raw["marital_status"].value_counts().sort_index()

marital_status
1.0    28537
3.0      866
4.0      413
5.0      262
Name: count, dtype: int64

In [4]:
flow = [("Ever-married women interviewed", len(raw))]

df = raw[raw["marital_status"] == 1].copy()
flow.append(("Currently married (V501 = 1)", len(df)))
len(df)

28537

In [5]:
df["fert_pref"].value_counts().sort_index()

fert_pref
1.0    2525
2.0    3466
3.0     240
4.0    1158
5.0    9894
6.0    1018
7.0     686
Name: count, dtype: int64

In [6]:
pd.crosstab(df["questionnaire"], df["fert_pref"].isna())

fert_pref,False,True
questionnaire,,
1.0,18987,0
2.0,0,9550


In [7]:
df["target"] = df["fert_pref"].map({1: "Non-Limiting", 2: "Non-Limiting", 3: "Non-Limiting", 4: "Non-Limiting",
                                    5: "Limiting", 6: "Limiting"})
df["undecided"] = (df["fert_pref"] == 4).astype(int)

In [8]:
df["target"].value_counts(dropna=False)

target
Limiting        10912
NaN             10236
Non-Limiting     7389
Name: count, dtype: int64

## Predictors

In [9]:
# Survey weight
df["weight"] = df["weight"] / 1_000_000

# Age group
df["age_cat"] = pd.cut(df["age"], bins=[14, 24, 34, 49],
                       labels=["15-24 (Younger)", "25-34 (Middle-aged)", "35-49 (Older)"]).astype(object)

# Division and place of residence
df["division_name"] = df["division"].map({1: "Barishal", 2: "Chattogram", 3: "Dhaka", 4: "Khulna",
                                          5: "Mymensingh", 6: "Rajshahi", 7: "Rangpur", 8: "Sylhet"})
df["residence_name"] = df["residence"].map({1: "Urban", 2: "Rural"})

In [10]:
years = df["years_in_residence"]
df["residence_duration"] = pd.cut(years, bins=[-0.5, 4.5, 14.5, 94.5],
                                  labels=["0-4 years", "5-14 years", "15+ years"]).astype(object)
df.loc[years == 95, "residence_duration"] = "Always resident"
df["residence_duration"].value_counts(dropna=False)


residence_duration
15+ years          9673
5-14 years         7904
0-4 years          5108
Always resident    4451
NaN                1401
Name: count, dtype: int64

In [11]:
years = df["years_since_cohabitation"]
df["marital_duration"] = pd.cut(years, bins=[-0.5, 4.5, 9.5, 14.5, 99],
                                labels=["0-4 years", "5-9 years", "10-14 years", "15+ years"]).astype(object)
df["marital_duration"].value_counts(dropna=False)


marital_duration
NaN            9550
15+ years      9056
0-4 years      3503
5-9 years      3262
10-14 years    3166
Name: count, dtype: int64

In [12]:
# Education of the respondent and husband (husband "don't know" grouped with no education)
df["edu_level_name"] = df["edu_level"].map({0: "No education", 1: "Primary", 2: "Secondary", 3: "Higher"})
df["husband_edu_cat"] = df["husband_edu"].map({0: "No education", 1: "Primary", 2: "Secondary",
                                               3: "Higher", 8: "No education"})

# Religion
df["religion_cat"] = df["religion"].map({1: "Muslim", 2: "Non-Muslim", 3: "Non-Muslim",
                                         4: "Non-Muslim", 96: "Non-Muslim"})

# Household size and sex of household head
df["household_size_cat"] = pd.cut(df["hh_size"], bins=[0.5, 3.5, 6.5, np.inf],
                                  labels=["Small (1-3)", "Medium (4-6)", "Large (7+)"]).astype(object)
df["hh_head_sex_cat"] = df["hh_head_sex"].map({1: "Male", 2: "Female"})

# Wealth index
df["wealth_cat"] = df["wealth_quintile"].map({1: "Poor", 2: "Poor", 3: "Middle", 4: "Rich", 5: "Rich"})

In [13]:
df["husband_edu"].value_counts(dropna=False).sort_index()


husband_edu
0.0    4011
1.0    5306
2.0    6092
3.0    3545
8.0      33
NaN    9550
Name: count, dtype: int64

In [14]:
# Media exposure: reads a newspaper, listens to the radio or watches television at all
media = df[["reads_newspaper", "listens_radio", "watches_tv"]]
exposed = media.isin([1, 2, 3]).any(axis=1)
known = exposed | media.notna().all(axis=1)
df["media_exposure"] = pd.Series(np.where(exposed, "Yes", "No"), index=df.index).where(known)

In [15]:
# Digital inclusion: internet use in the last 12 months, smartphone ownership, bank or mobile-money
# account, and account use in the last 12 months. Smartphone ownership is asked only of mobile-phone
# owners and account use only of account holders; skipped items are scored as absent.
df["dig_internet"] = (df["used_internet_12m"] == 1).astype(int).where(df["used_internet_12m"].notna())
df["dig_smartphone"] = (df["is_smartphone"].mask(df["owns_mobile"] == 0, 0) == 1).astype(int)
df["dig_smartphone"] = df["dig_smartphone"].where(df["is_smartphone"].notna() | (df["owns_mobile"] == 0))
df["dig_account"] = (df["bank_account"] == 1).astype(int).where(df["bank_account"].notna())
df["dig_account_use"] = (df["bank_transaction_12m"].mask(df["bank_account"] == 0, 0) == 1).astype(int)
df["dig_account_use"] = df["dig_account_use"].where(df["bank_transaction_12m"].notna() | (df["bank_account"] == 0))

df["digital_score"] = df[["dig_internet", "dig_smartphone", "dig_account", "dig_account_use"]].sum(axis=1, min_count=4)
df["digital_inclusion_cat"] = pd.cut(df["digital_score"], bins=[-0.5, 0.5, 2.5, 4.5],
                                     labels=["Low", "Medium", "High"]).astype(object)
df["digital_score"].value_counts(dropna=False).sort_index()

digital_score
0.0    10786
1.0     2201
2.0     3722
3.0      768
4.0     1510
NaN     9550
Name: count, dtype: int64

In [16]:
# Experience of child death
deaths = df[["sons_died", "daughters_died"]].sum(axis=1, min_count=2)
df["child_death_experience"] = pd.Series(np.where(deaths > 0, "Yes", "No"), index=df.index).where(deaths.notna())

# Currently pregnant
df["is_pregnant"] = df["currently_pregnant"].map({0: "No", 1: "Yes"})

# Number of living children
df["living_children_cat"] = pd.cut(df["living_children"], bins=[-0.5, 0.5, 3.5, np.inf],
                                   labels=["0 children", "1-3 children", "4+ children"]).astype(object)

In [17]:
# Contraceptive behavior and intention (V312, V362)
method, intent = df["contra_method"], df["intent_use_contra"]
df["contra_behavior_intent"] = pd.Series(np.nan, index=df.index, dtype=object)
df.loc[(method == 0) & intent.notna(), "contra_behavior_intent"] = "Non-user: Unsure/Other"
df.loc[(method == 0) & intent.isin([1, 2]), "contra_behavior_intent"] = "Non-user: Intends to use"
df.loc[(method == 0) & (intent == 5), "contra_behavior_intent"] = "Non-user: No intent"
df.loc[method >= 1, "contra_behavior_intent"] = "Currently Using"

# Sterilization (female V312 = 6, male V312 = 7) and method type
df["sterilized"] = method.isin([6, 7]).astype(int)
df["contra_method_type"] = np.select([method.isin([6, 7]), method >= 1, method == 0],
                                     ["Permanent", "Reversible", "None"], default="")
df["contra_method_type"] = df["contra_method_type"].replace("", np.nan)

pd.crosstab(df["contra_method"], df["fert_pref"])


fert_pref,1.0,2.0,3.0,4.0,5.0,6.0,7.0
contra_method,,,,,,,
0.0,1760,1078,149,522,2606,0,627
1.0,371,1202,51,324,3286,0,20
2.0,2,10,1,2,75,0,2
3.0,82,413,8,102,1492,0,9
5.0,156,455,13,107,869,0,5
6.0,0,0,0,0,0,839,0
7.0,0,0,0,0,0,179,0
8.0,76,116,13,45,931,0,12
9.0,63,139,4,43,348,0,8


In [18]:
# Husband's residence (V504)
df["husband_presence_cat"] = df["husband_lives_with"].map({1: "Living with her", 2: "Living elsewhere",
                                                           3: "Living elsewhere"})

# Age at first marriage
df["marriage_age_cat"] = pd.cut(df["age_first_marriage"], bins=[-np.inf, 15, 18, np.inf], right=False,
                                labels=["<15 years", "15-17 years", "18+ years"]).astype(object)

# Ideal number of children (V613): 96 = non-numeric answer, 99 = missing
ideal = df["ideal_children"].replace({96: np.nan, 99: np.nan})
df["ideal_family_size_cat"] = pd.Series(np.where(ideal <= 2, "0-2 Children (Small)", "3+ Children (Large)"),
                                        index=df.index).where(ideal.notna())

In [19]:
# Husband's occupation (V704)
occ = df["husband_occupation"]
df["husband_occ_cat"] = pd.Series(
    np.select([occ.between(11, 16), occ.between(21, 23), occ == 31, occ == 41, occ.isin([51, 52])],
              ["Agriculture", "Manual Labor", "Skilled Trade/Service", "Professional/Technical", "Business/Trade"],
              default="Not working/Other"),
    index=df.index).where(occ.notna())

# Respondent currently working (V714)
df["working_status"] = df["currently_working"].map({1: "Yes (Working)", 0: "No (Not Working)"})

df["husband_occ_cat"].value_counts(dropna=False)


husband_occ_cat
NaN                       9550
Manual Labor              5528
Agriculture               4486
Business/Trade            3840
Skilled Trade/Service     3033
Professional/Technical    1328
Not working/Other          772
Name: count, dtype: int64

In [20]:
# Decision-making autonomy: number of three decisions made alone or jointly with the husband
dec = df[["dec_own_health", "dec_large_purchase", "dec_visit_family"]]
autonomy = dec.isin([1, 2]).sum(axis=1).where(dec.notna().all(axis=1))
df["women_autonomy"] = pd.cut(autonomy, bins=[-0.5, 0.5, 2.5, 3.5],
                              labels=["Low Autonomy", "Medium Autonomy", "High Autonomy"]).astype(object)

# Wife beating justified for at least one of five reasons
beat = df[["beat_goes_out", "beat_neglects_kids", "beat_argues", "beat_refuses_sex", "beat_burns_food"]]
justifies = (beat == 1).any(axis=1)
known = justifies | beat.notna().all(axis=1)
df["justifies_violence"] = pd.Series(np.where(justifies, "Yes", "No"), index=df.index).where(known)

In [21]:
# Anxiety (GAD-7) and depression (PHQ-9). "Refused" (7) and "don't know" (8) are missing;
# a scale is scored only when all items are answered.
gad = df[GAD_COLS].replace({7: np.nan, 8: np.nan})
phq = df[PHQ_COLS].replace({7: np.nan, 8: np.nan})
df["gad_score"] = gad.sum(axis=1).where(gad.notna().all(axis=1))
df["phq_score"] = phq.sum(axis=1).where(phq.notna().all(axis=1))

df["anxiety_severity"] = pd.cut(df["gad_score"], bins=[-0.5, 4.5, 9.5, 21.5],
                                labels=["Minimal (0-4)", "Mild (5-9)", "Moderate/Severe (10-21)"]).astype(object)
df["depression_severity"] = pd.cut(df["phq_score"], bins=[-0.5, 4.5, 9.5, 27.5],
                                   labels=["Minimal (0-4)", "Mild (5-9)", "Moderate/Severe (10-27)"]).astype(object)

In [22]:
CAT_ORDER = {
    "target": ["Limiting", "Non-Limiting"],
    "age_cat": ["15-24 (Younger)", "25-34 (Middle-aged)", "35-49 (Older)"],
    "division_name": ["Barishal", "Chattogram", "Dhaka", "Khulna", "Mymensingh", "Rajshahi", "Rangpur", "Sylhet"],
    "residence_name": ["Urban", "Rural"],
    "residence_duration": ["Always resident", "15+ years", "5-14 years", "0-4 years"],
    "marital_duration": ["0-4 years", "5-9 years", "10-14 years", "15+ years"],
    "edu_level_name": ["No education", "Primary", "Secondary", "Higher"],
    "religion_cat": ["Muslim", "Non-Muslim"],
    "household_size_cat": ["Small (1-3)", "Medium (4-6)", "Large (7+)"],
    "hh_head_sex_cat": ["Male", "Female"],
    "media_exposure": ["Yes", "No"],
    "wealth_cat": ["Poor", "Middle", "Rich"],
    "digital_inclusion_cat": ["Low", "Medium", "High"],
    "child_death_experience": ["No", "Yes"],
    "is_pregnant": ["No", "Yes"],
    "living_children_cat": ["0 children", "1-3 children", "4+ children"],
    "contra_behavior_intent": ["Currently Using", "Non-user: Intends to use",
                               "Non-user: No intent", "Non-user: Unsure/Other"],
    "husband_presence_cat": ["Living with her", "Living elsewhere"],
    "marriage_age_cat": ["<15 years", "15-17 years", "18+ years"],
    "ideal_family_size_cat": ["0-2 Children (Small)", "3+ Children (Large)"],
    "husband_edu_cat": ["No education", "Primary", "Secondary", "Higher"],
    "husband_occ_cat": ["Not working/Other", "Agriculture", "Manual Labor", "Skilled Trade/Service",
                        "Business/Trade", "Professional/Technical"],
    "working_status": ["No (Not Working)", "Yes (Working)"],
    "women_autonomy": ["Low Autonomy", "Medium Autonomy", "High Autonomy"],
    "justifies_violence": ["No", "Yes"],
    "anxiety_severity": ["Minimal (0-4)", "Mild (5-9)", "Moderate/Severe (10-21)"],
    "depression_severity": ["Minimal (0-4)", "Mild (5-9)", "Moderate/Severe (10-27)"],
}
ANALYSIS_VARS = [c for c in CAT_ORDER if c != "marital_duration"]

for col, order in CAT_ORDER.items():
    unexpected = set(df[col].dropna().unique()) - set(order)
    assert not unexpected, f"{col}: {unexpected}"
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)


## Analytic sample

In [23]:
long_q = df["questionnaire"] == 1
step1 = long_q
flow.append(("Long questionnaire (SQTYPE = 1)", step1.sum()))

# Mental-health items administered to long-questionnaire women
print("GAD-7 item 1 answered:", df.loc[long_q, "gad_1"].notna().sum(), "of", long_q.sum())
print("PHQ-9 item 1 answered:", df.loc[long_q, "phq_1"].notna().sum(), "of", long_q.sum())

GAD-7 item 1 answered: 18987 of 18987
PHQ-9 item 1 answered: 18987 of 18987


In [24]:
step2 = step1 & df["fert_pref"].notna() & (df["fert_pref"] != 7)
flow.append(("Fertility preference classifiable (V605 not 7 or missing)", step2.sum()))

step3 = step2 & (df["years_in_residence"] != 96)
flow.append(("Usual residents (V104 not 96, visitor)", step3.sum()))

missing = df.loc[step3, ANALYSIS_VARS].isna().sum()
missing[missing > 0]

ideal_family_size_cat    74
anxiety_severity         15
depression_severity      38
dtype: int64

In [25]:
step4 = step3 & df[ANALYSIS_VARS].notna().all(axis=1)
flow.append(("Complete data on all analysis variables", step4.sum()))

df["included"] = step4.astype(int)

flow_table = pd.DataFrame(flow, columns=["step", "n"])
flow_table["excluded"] = -flow_table["n"].diff()
flow_table

,step,n,excluded
0,Ever-married women interviewed,30078,NaN
1,Currently married (V501 = 1),28537,1541.0
2,Long questionnaire (SQTYPE = 1),18987,9550.0
3,Fertility preference classifiable (V605 not 7 ...,18301,686.0
4,"Usual residents (V104 not 96, visitor)",17368,933.0
5,Complete data on all analysis variables,17245,123.0


In [26]:
analytic = df[df["included"] == 1].copy()

summary = pd.DataFrame({
    "n": analytic["target"].value_counts(),
    "weighted %": analytic.groupby("target", observed=False)["weight"].sum() / analytic["weight"].sum() * 100,
})
summary.round(1)

,n,weighted %
target,,
Limiting,10566,60.9
Non-Limiting,6679,39.1


In [27]:
# Composition of the outcome categories
pd.DataFrame({
    "undecided (V605 = 4)": [analytic["undecided"].sum()],
    "sterilized (V605 = 6)": [(analytic["fert_pref"] == 6).sum()],
    "female sterilization (V312 = 6)": [(analytic["contra_method"] == 6).sum()],
    "male sterilization (V312 = 7)": [(analytic["contra_method"] == 7).sum()],
})

,undecided (V605 = 4),sterilized (V605 = 6),female sterilization (V312 = 6),male sterilization (V312 = 7)
0,1069,998,822,176


In [28]:
from pathlib import Path

OUT = Path("../output")

KEEP = ["psu", "strata", "weight", "target", "fert_pref", "undecided", "sterilized",
        "contra_method", "contra_method_type", "years_in_residence", "previous_residence",
        "reason_moved", "years_since_cohabitation", "dig_internet", "dig_smartphone",
        "dig_account", "dig_account_use", "owns_mobile", "digital_score", "gad_score",
        "phq_score", "age", "living_children", "marital_duration"]
KEEP = list(dict.fromkeys(KEEP + ANALYSIS_VARS))

analytic[KEEP].to_csv(OUT / "dataset.csv", index=False)
flow_table.to_csv(OUT / "sample_flow.csv", index=False)


## Reliability of the scales

In [29]:
dig = analytic[["dig_internet", "dig_smartphone", "dig_account", "dig_account_use"]]
gad = analytic[GAD_COLS]
phq = analytic[PHQ_COLS]

def reliability(items):
    k = items.shape[1]
    return k / (k - 1) * (1 - items.var(ddof=1).sum() / items.sum(axis=1).var(ddof=1))

pd.Series({"Digital inclusion index (KR-20)": reliability(dig),
           "GAD-7 (Cronbach alpha)": reliability(gad),
           "PHQ-9 (Cronbach alpha)": reliability(phq)}).round(2)


Digital inclusion index (KR-20)    0.76
GAD-7 (Cronbach alpha)             0.82
PHQ-9 (Cronbach alpha)             0.81
dtype: float64

## Included and excluded women

In [30]:
background = ["age_cat", "residence_name", "division_name", "edu_level_name",
              "wealth_cat", "religion_cat", "living_children_cat"]

rows = []
for var in background:
    w = df.groupby(["included", var], observed=True)["weight"].sum().unstack(0)
    pct = w / w.sum() * 100
    pct.index = pd.MultiIndex.from_product([[var], pct.index])
    rows.append(pct)

comparison = pd.concat(rows).rename(columns={0: "Excluded (%)", 1: "Included (%)"}).round(1)
comparison


included                                 Excluded (%)  Included (%)
age_cat             15-24 (Younger)              26.4          25.0
                    25-34 (Middle-aged)          33.1          36.5
                    35-49 (Older)                40.4          38.5
residence_name      Urban                        27.9          28.6
                    Rural                        72.1          71.4
division_name       Barishal                      6.4           5.9
                    Chattogram                   18.3          18.6
                    Dhaka                        24.8          25.7
                    Khulna                       12.1          11.9
                    Mymensingh                    8.0           7.5
                    Rajshahi                     13.1          13.2
                    Rangpur                      11.5          11.7
                    Sylhet                        5.7           5.5
edu_level_name      No education                 14.3          12.5
                    Primary                      25.3          26.2
                    Secondary                    45.6          47.8
                    Higher                       14.8          13.5
wealth_cat          Poor                         39.1          37.7
                    Middle                       20.3          20.5
                    Rich                         40.6          41.7
religion_cat        Muslim                       90.3          90.1
                    Non-Muslim                    9.7           9.9
living_children_cat 0 children                   11.7           9.9
                    1-3 children                 77.8          79.7
                    4+ children                  10.5          10.4

## Table 1

In [31]:
rows = []
for var in [v for v in ANALYSIS_VARS if v != "target"]:
    w = analytic.groupby(var, observed=True)["weight"].sum()
    wl = analytic.groupby([var, "target"], observed=True)["weight"].sum().unstack()
    table = pd.DataFrame({
        "n": analytic[var].value_counts(),
        "Weighted %": w / w.sum() * 100,
        "Limiting (weighted %)": wl["Limiting"] / wl.sum(axis=1) * 100,
    })
    table.index = pd.MultiIndex.from_product([[var], table.index])
    rows.append(table)

table1 = pd.concat(rows).round(1)
table1


n  Weighted %  \
age_cat             15-24 (Younger)           4208        25.0   
                    25-34 (Middle-aged)       6314        36.5   
                    35-49 (Older)             6723        38.5   
division_name       Barishal                  1816         5.9   
                    Chattogram                2570        18.6   
...                                            ...         ...   
anxiety_severity    Mild (5-9)                3751        21.5   
                    Moderate/Severe (10-21)    695         4.0   
depression_severity Minimal (0-4)            12354        72.0   
                    Mild (5-9)                4067        23.4   
                    Moderate/Severe (10-27)    824         4.6   

                                             Limiting (weighted %)  
age_cat             15-24 (Younger)                           17.5  
                    25-34 (Middle-aged)                       59.0  
                    35-49 (Older)                             91.0  
division_name       Barishal                                  61.5  
                    Chattogram                                57.6  
...                                                            ...  
anxiety_severity    Mild (5-9)                                68.2  
                    Moderate/Severe (10-21)                   71.5  
depression_severity Minimal (0-4)                             58.9  
                    Mild (5-9)                                65.4  
                    Moderate/Severe (10-27)                   68.8  

[77 rows x 3 columns]

In [32]:
from scipy import stats

def rao_scott(data, var, outcome="target"):
    d = data[[var, outcome, "weight", "psu", "strata"]].dropna()
    rows = pd.Categorical(d[var]).codes.astype(int)
    cols = pd.Categorical(d[outcome]).codes.astype(int)
    R, C = int(rows.max()) + 1, int(cols.max()) + 1
    n, K = len(d), R * C

    Y = np.zeros((n, K))
    Y[np.arange(n), rows + R * cols] = 1
    w = d["weight"].to_numpy()
    p = (w[:, None] * Y).sum(axis=0) / w.sum()

    # design-based covariance of the cell proportions (strata and primary sampling units)
    u = w[:, None] * (Y - p) / w.sum()
    z = pd.DataFrame(u).groupby([d["strata"].to_numpy(), d["psu"].to_numpy()]).sum()
    V = np.zeros((K, K))
    for _, zh in z.groupby(level=0):
        nh = len(zh)
        zc = zh.to_numpy() - zh.to_numpy().mean(axis=0)
        V += nh / (nh - 1) * zc.T @ zc

    cell_row, cell_col = np.tile(np.arange(R), C), np.repeat(np.arange(C), R)
    X1 = np.column_stack([np.ones(K)] + [(cell_row == i).astype(float) for i in range(1, R)]
                         + [(cell_col == j).astype(float) for j in range(1, C)])
    X12 = np.column_stack([((cell_row == i) & (cell_col == j)).astype(float)
                           for i in range(1, R) for j in range(1, C)])
    Cm = X12 - X1 @ np.linalg.lstsq(X1, X12, rcond=None)[0]
    iD = np.diag(1 / p)
    Delta = np.linalg.solve(Cm.T @ iD @ Cm / n, Cm.T @ iD @ V @ iD @ Cm)

    P = p.reshape(C, R).T
    expected = np.outer(P.sum(axis=1), P.sum(axis=0))
    chi2 = n * ((P - expected) ** 2 / expected).sum()
    df = (R - 1) * (C - 1)
    chi2_rs = chi2 * df / np.trace(Delta)
    return {"Rao-Scott chi2": chi2_rs, "df": df, "p": stats.chi2.sf(chi2_rs, df),
            "Cramer's V": np.sqrt(chi2 / (n * min(R - 1, C - 1)))}

tests = pd.DataFrame({v: rao_scott(analytic, v) for v in ANALYSIS_VARS if v != "target"}).T
tests["p"] = tests["p"].map(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")

final = pd.DataFrame(index=table1.index)
final["n (%)"] = [f"{n:,} ({p:.1f})" for n, p in zip(table1["n"], table1["Weighted %"])]
final["Limiting (%)"] = table1["Limiting (weighted %)"]
final["Rao-Scott chi2 (df)"] = ""
final["p"] = ""
final["Cramer's V"] = ""

for var, t in tests.iterrows():
    first = (var, final.loc[var].index[0])
    final.loc[first, "Rao-Scott chi2 (df)"] = f"{t['Rao-Scott chi2']:.2f} ({int(t['df'])})"
    final.loc[first, "p"] = t["p"]
    final.loc[first, "Cramer's V"] = "{:.3f}".format(t["Cramer's V"])


final.to_csv("../output/table1.csv")
final

n (%)  Limiting (%)  \
age_cat             15-24 (Younger)           4,208 (25.0)          17.5   
                    25-34 (Middle-aged)       6,314 (36.5)          59.0   
                    35-49 (Older)             6,723 (38.5)          91.0   
division_name       Barishal                   1,816 (5.9)          61.5   
                    Chattogram                2,570 (18.6)          57.6   
...                                                    ...           ...   
anxiety_severity    Mild (5-9)                3,751 (21.5)          68.2   
                    Moderate/Severe (10-21)      695 (4.0)          71.5   
depression_severity Minimal (0-4)            12,354 (72.0)          58.9   
                    Mild (5-9)                4,067 (23.4)          65.4   
                    Moderate/Severe (10-27)      824 (4.6)          68.8   

                                            Rao-Scott chi2 (df)       p  \
age_cat             15-24 (Younger)                 4265.89 (2)  <0.001   
                    25-34 (Middle-aged)                                   
                    35-49 (Older)                                         
division_name       Barishal                          57.33 (7)  <0.001   
                    Chattogram                                            
...                                                         ...     ...   
anxiety_severity    Mild (5-9)                                            
                    Moderate/Severe (10-21)                               
depression_severity Minimal (0-4)                     63.46 (2)  <0.001   
                    Mild (5-9)                                            
                    Moderate/Severe (10-27)                               

                                            Cramer's V  
age_cat             15-24 (Younger)              0.588  
                    25-34 (Middle-aged)                 
                    35-49 (Older)                       
division_name       Barishal                     0.061  
                    Chattogram                          
...                                                ...  
anxiety_severity    Mild (5-9)                          
                    Moderate/Severe (10-21)             
depression_severity Minimal (0-4)                0.067  
                    Mild (5-9)                          
                    Moderate/Severe (10-27)             

[77 rows x 5 columns]

## Residence duration within strata

In [33]:
limiting = analytic["target"] == "Limiting"
groups = [analytic["marital_duration"], analytic["residence_duration"]]

limiting_pct = (analytic["weight"].where(limiting, 0).groupby(groups, observed=True).sum()
                / analytic["weight"].groupby(groups, observed=True).sum() * 100).unstack()
n_women = analytic.groupby(["marital_duration", "residence_duration"], observed=True).size().unstack()

print("Limiting preference (weighted %)")
display(limiting_pct.round(1))
print("Number of women")
display(n_women)
limiting_pct.round(1).to_csv("../output/stratified_residence_by_marital_duration.csv")


Limiting preference (weighted %)


residence_duration,Always resident,15+ years,5-14 years,0-4 years
marital_duration,,,,
0-4 years,6.9,57.0,21.4,5.5
5-9 years,34.8,37.1,35.4,31.9
10-14 years,59.8,71.5,62.6,57.9
15+ years,90.0,91.0,83.4,80.1


Number of women


residence_duration,Always resident,15+ years,5-14 years,0-4 years
marital_duration,,,,
0-4 years,566,64,212,2193
5-9 years,432,51,2151,364
10-14 years,436,202,2019,328
15+ years,1388,5593,793,453


In [34]:
limiting = analytic["target"] == "Limiting"

for var in ["age_cat", "living_children_cat"]:
    groups = [analytic[var], analytic["residence_duration"]]
    pct = (analytic["weight"].where(limiting, 0).groupby(groups, observed=True).sum()
           / analytic["weight"].groupby(groups, observed=True).sum() * 100).unstack()
    n_women = analytic.groupby([var, "residence_duration"], observed=True).size().unstack()
    display(pct.round(1))
    display(n_women)
    pct.round(1).to_csv(f"../output/stratified_residence_by_{var}.csv")

residence_duration,Always resident,15+ years,5-14 years,0-4 years
age_cat,,,,
15-24 (Younger),18.4,19.2,32.6,7.8
25-34 (Middle-aged),57.1,79.5,55.0,40.9
35-49 (Older),90.8,93.3,81.9,80.2


residence_duration,Always resident,15+ years,5-14 years,0-4 years
age_cat,,,,
15-24 (Younger),731,42,1309,2126
25-34 (Middle-aged),943,1400,3125,846
35-49 (Older),1148,4468,741,366


residence_duration,Always resident,15+ years,5-14 years,0-4 years
living_children_cat,,,,
0 children,2.7,5.0,1.9,1.5
1-3 children,65.3,88.5,53.3,31.8
4+ children,97.7,98.1,95.9,99.0


residence_duration,Always resident,15+ years,5-14 years,0-4 years
living_children_cat,,,,
0 children,376,72,191,1068
1-3 children,2134,4562,4802,2205
4+ children,312,1276,182,65


## Previous residence and reason for moving

In [35]:
moved = analytic[analytic["residence_duration"] != "Always resident"]

previous = {0: "Capital/large city", 1: "City", 2: "Town", 3: "Countryside", 4: "Abroad"}
reason = {1: "Employment", 2: "Education/training", 3: "Marriage", 4: "Other family reason",
          5: "Forced displacement", 96: "Other"}

rows = []
for var, labels in [("previous_residence", previous), ("reason_moved", reason)]:
    group = moved[var].map(labels)
    w = moved["weight"].groupby(group).sum()
    wl = moved["weight"].where(moved["target"] == "Limiting", 0).groupby(group).sum()
    t = pd.DataFrame({"n": group.value_counts(), "Weighted %": w / w.sum() * 100,
                      "Limiting (weighted %)": wl / w * 100})
    t.index = pd.MultiIndex.from_product([[var], t.index])
    rows.append(t)

migration_detail = pd.concat(rows).round(1)
migration_detail.to_csv("../output/migration_detail.csv")
migration_detail


n  Weighted %  \
previous_residence Capital/large city     907         5.7   
                   Countryside           1363         8.6   
                   Town                 12153        85.6   
reason_moved       Education/training      80         0.5   
                   Employment             582         4.7   
                   Forced displacement      6         0.1   
                   Marriage             11678        80.3   
                   Other                   59         0.4   
                   Other family reason   2018        14.0   

                                        Limiting (weighted %)  
previous_residence Capital/large city                    58.0  
                   Countryside                           60.3  
                   Town                                  61.3  
reason_moved       Education/training                    61.8  
                   Employment                            59.2  
                   Forced displacement                   54.6  
                   Marriage                              61.2  
                   Other                                 50.4  
                   Other family reason                   60.8

## Save category order

In [36]:
import json
json.dump({k: list(v) for k, v in CAT_ORDER.items()}, open(OUT / "cat_order.json", "w"))
